In [ ]:
helm list -n kube-system
helm uninstall 

In [ ]:
cat > pv.yaml << 'OF'
apiVersion: v1
kind: PersistentVolume
metadata:
  name: clickhouse-iscsi-pv
spec:
  storageClassName: "manual-block" # Explicit matching class
  capacity:
    storage: 20Gi
  volumeMode: Block
  accessModes:
    - ReadWriteOnce
  persistentVolumeReclaimPolicy: Retain
  local:
    path: /dev/sdb
  nodeAffinity:
    required:
      nodeSelectorTerms:
      - matchExpressions:
        - key: kubernetes.io/hostname
          operator: In
          values:
          - worker-1
          - worker-2
          - worker-3
OF
kubectl apply -f pv.yaml

In [ ]:
cat > clickhouse-deployment.yaml << 'OF'
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: clickhouse
spec:
  serviceName: "clickhouse"
  replicas: 1
  selector:
    matchLabels:
      app: clickhouse
  template:
    metadata:
      labels:
        app: clickhouse
    spec:
      containers:
      - name: clickhouse
        image: clickhouse/clickhouse-server:latest
        ports:
        - containerPort: 8123
          name: http
        - containerPort: 9000
          name: client
        volumeDevices:
        - name: clickhouse-storage
          devicePath: /dev/clickhouse-disk
  volumeClaimTemplates:
  - metadata:
      name: clickhouse-storage
    spec:
      storageClassName: "manual-block" # Matches the PV exactly
      accessModes: [ "ReadWriteOnce" ]
      volumeMode: Block
      resources:
        requests:
          storage: 20Gi
OF
kubectl apply -f clickhouse-deployment.yaml

In [ ]:
kubectl get pv,pvc,pods -o wide

---

On all worker

In [ ]:
# For RHEL/Rocky Linux 9 (matches your project baseline)
yum install -y iscsi-initiator-utils device-mapper-multipath

Eliminate Manual Node Path Configs
  
Instead of editing local-path-config Map files by hand to bind specific node paths, you deploy a dynamic StorageClass. This represents your PowerVault storage poo

In [ ]:
cat > dynamic-storageclass.yaml <<OF
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata:
  name: simulated-powervault-xfs
provisioner: org.democratic-csi.iscsi-generic
volumeBindingMode: WaitForFirstConsumer
parameters:
  fsType: xfs
OF

kubectl apply -f dynamic-storageclass.yaml

# apiVersion: storage.k8s.io/v1
# kind: StorageClass
# metadata:
#   name: simulated-powervault-xfs  # Simulates your Dell PowerVault pool
# provisioner: kubernetes.io/no-provisioner  # We use the local provisioner framework for lab simulation
# volumeBindingMode: WaitForFirstConsumer  # CRITICAL: Waits until the pod is scheduled before binding
# allowVolumeExpansion: true
# parameters:
#   fsType: xfs  # Forces the cluster to automatically format the volume as XFS

Let Your Applications Request Storage Dynamically

Your StatefulSets or PersistentVolumeClaims no longer need pre-prepared directories on the host operating systems. They simply request the storage class directly:

In [ ]:
cat > xdr-data-volume.yaml << OF
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: xdr-data-volume
spec:
  accessModes:
    - ReadWriteOnce
  storageClassName: "simulated-powervault-xfs" # <-- Requests the driver to take action
  resources:
    requests:
      storage: 20Gi
OF

kubectl apply -f  xdr-data-volume.yaml

In [ ]:
kubectl get nodes --show-labels
kubectl get nodes -L workload-class
kubectl label nodes worker-1 worker-2 worker-3 workload-class=heavy

In [ ]:
cat > clickhouse-xfs-test.yaml <<OF
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: clickhouse-xfs
spec:
  serviceName: "clickhouse-test"
  replicas: 1
  selector:
    matchLabels:
      app: clickhouse-xfs
  template:
    metadata:
      labels:
        app: clickhouse-xfs
    spec:
      nodeSelector:
        workload-class: "heavy" 
      containers:
      - name: clickhouse
        image: clickhouse/clickhouse-server:latest
        ports:
        - containerPort: 8123
          name: http
        volumeMounts:
        - name: xdr-data-volume
          mountPath: /var/lib/clickhouse
  volumeClaimTemplates:
  - metadata:
      name: xdr-data-volume
    spec:
      accessModes: [ "ReadWriteOnce" ]
      storageClassName: "simulated-powervault-xfs"
      resources:
        requests:
          storage: 20Gi
OF

kubectl apply -f clickhouse-xfs-test.yaml

In [ ]:

cat > democratic-csi-values.yaml <<'OF'
csiDriver:
  name: "org.democratic-csi.iscsi-generic"

controller:
  enabled: true
  strategy: deployment
  replicaCount: 1
  externalProvisioner:
    enabled: true
    image:
      registry: registry.k8s.io
      repository: sig-storage/csi-provisioner
      tag: v4.0.1
  # Explicitly disable the crashing sidecars
  externalResizer:
    enabled: false
  externalSnapshotter:
    enabled: false

node:
  enabled: true

driver:
  config:
    driver: freenas-api
    httpConnection:
      protocol: http
      host: 127.0.0.1
      allowInsecure: true
      apiKey: "dummy"
    
    node-manual:
      driver: node-manual
      iscsi:
        targetPortal: "172.16.6.69:3260" 
        targetPortals: []
        namePrefix: "iqn.2026-04.local.lab:"
        interface: ""
      ssh:
        host: "172.16.6.69"
        port: 22
        username: "root"
        password: "123456"
      driverOptions:
        cli:
          createVolume: "targetcli /backstores/fileio create name={{volume.name}} size={{volume.size}} file_or_dev=/var/lib/iscsi-images/{{volume.name}}.img"
          deleteVolume: "targetcli /backstores/fileio delete name={{volume.name}}"
          getVolume: "targetcli /backstores/fileio info name={{volume.name}}"
OF

In [ ]:
# 1. Purge old helm remnants completely and upgrade
helm upgrade democratic-csi-iscsi democratic-csi/democratic-csi \
  --namespace kube-system \
  --values democratic-csi-values.yaml

# 2. Reset the ClickHouse state machine requests
kubectl delete statefulset clickhouse-xfs --ignore-not-found=true
kubectl delete pvc xdr-data-volume-clickhouse-xfs-0 --ignore-not-found=true

# 3. Fire up the ClickHouse service
kubectl apply -f clickhouse-xfs-test.yaml

In [ ]:
kubectl get pods -n kube-system -l app.kubernetes.io/name=democratic-csi

In [ ]:
# cat > democratic-csi-values.yaml <<OF
# csiDriver:
#   name: "org.democratic-csi.iscsi-generic"

# # This block houses all driver configuration parameters
# driver:
#   config:
#     driver: iscsi-generic

#     # iSCSI configurations for your PowerVault SAN network
#     iscsi:
#       targetPortal: "172.16.6.69:3260" 
#       targetPortals: []
#       namePrefix: "iqn.2026-04.local.lab:"
#       interface: ""

#     # How the CSI driver communicates with the PowerVault management interface
#     ssh:
#       host: "172.16.6.69"
#       port: 22
#       username: "root"
#       password: "123456"

#     # Storage provisioning mechanics
#     driverOptions:
#       cli:
#         createVolume: "volume create name {{volume.name}} size {{volume.size}} pool production-pool"
#         deleteVolume: "volume delete name {{volume.name}}"
#         getVolume: "volume show name {{volume.name}}"
# OF

In [ ]:
# 1. Add and update the Helm repository
helm repo add democratic-csi https://democratic-csi.github.io/charts/
helm repo update

# 2. Install the driver using your values file
helm install democratic-csi-iscsi democratic-csi/democratic-csi \
  --namespace kube-system \
  --values democratic-csi-values.yaml

In [ ]:
helm upgrade democratic-csi-iscsi democratic-csi/democratic-csi \
  --namespace kube-system \
  --values democratic-csi-values.yaml

In [ ]:
kubectl get pods -n kube-system -l app.kubernetes.io/name=democratic-csi -o wide
kubectl get pods -n kube-system -l app.kubernetes.io/name=democratic-csi

---

In [ ]:
# 1. Clean up old stuck components completely
kubectl delete statefulset clickhouse-xfs --ignore-not-found=true
kubectl delete pvc xdr-data-volume-clickhouse-xfs-0 --ignore-not-found=true
kubectl delete pvc xdr-data-volume --ignore-not-found=true

# 2. Create the StorageClass
cat > dynamic-storageclass.yaml <<OF
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata:
  name: simulated-powervault-xfs
provisioner: org.democratic-csi.iscsi-generic
volumeBindingMode: WaitForFirstConsumer
parameters:
  fsType: xfs
OF

kubectl apply -f dynamic-storageclass.yaml

# 3. Create the ClickHouse StatefulSet (Standalone PVC removed, NodeSelector commented out for safety)
cat > clickhouse-xfs-test.yaml <<OF
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: clickhouse-xfs
spec:
  serviceName: "clickhouse-test"
  replicas: 1
  selector:
    matchLabels:
      app: clickhouse-xfs
  template:
    metadata:
      labels:
        app: clickhouse-xfs
    spec:
      # Commented out to ensure it schedules on your available lab workers smoothly
      # nodeSelector:
      #   workload-class: "heavy" 
      containers:
      - name: clickhouse
        image: clickhouse/clickhouse-server:latest
        ports:
        - containerPort: 8123
          name: http
        volumeMounts:
        - name: xdr-data-volume
          mountPath: /var/lib/clickhouse
  volumeClaimTemplates:
  - metadata:
      name: xdr-data-volume
    spec:
      accessModes: [ "ReadWriteOnce" ]
      storageClassName: "simulated-powervault-xfs"
      resources:
        requests:
          storage: 20Gi
OF

kubectl apply -f clickhouse-xfs-test.yaml

In [ ]:
cat > standalone-csi-provisioner.yaml << 'OF'
apiVersion: v1
kind: ServiceAccount
metadata:
  name: democratic-csi-external-provisioner
  namespace: kube-system
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: democratic-csi-external-provisioner-role
rules:
  - apiGroups: [""]
    resources: ["persistentvolumeclaims"]
    verbs: ["get", "list", "watch", "update"]
  - apiGroups: [""]
    resources: ["persistentvolumes"]
    verbs: ["get", "list", "watch", "create", "delete", "update"]
  - apiGroups: ["storage.k8s.io"]
    resources: ["storageclasses"]
    verbs: ["get", "list", "watch"]
  - apiGroups: [""]
    resources: ["events"]
    verbs: ["list", "watch", "create", "update", "patch"]
  - apiGroups: ["snapshot.storage.k8s.io"]
    resources: ["volumesnapshots"]
    verbs: ["get", "list"]
  - apiGroups: ["snapshot.storage.k8s.io"]
    resources: ["volumesnapshotcontents"]
    verbs: ["get", "list"]
  - apiGroups: ["storage.k8s.io"]
    resources: ["csinodes"]
    verbs: ["get", "list", "watch"]
  - apiGroups: [""]
    resources: ["nodes"]
    verbs: ["get", "list", "watch"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: democratic-csi-external-provisioner-binding
subjects:
  - kind: ServiceAccount
    name: democratic-csi-external-provisioner
    namespace: kube-system
roleRef:
  kind: ClusterRole
  name: democratic-csi-external-provisioner-role
  apiGroup: rbac.authorization.k8s.io
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: democratic-csi-standalone-provisioner
  namespace: kube-system
spec:
  replicas: 1
  selector:
    matchLabels:
      app: democratic-csi-standalone-provisioner
  template:
    metadata:
      labels:
        app: democratic-csi-standalone-provisioner
    spec:
      serviceAccountName: democratic-csi-external-provisioner
      containers:
        - name: csi-provisioner
          image: registry.k8s.io/sig-storage/csi-provisioner:v4.0.1
          args:
            - "--v=5"
            - "--csi-address=/csi-data/csi.sock"
            - "--feature-gates=Topology=true"
            - "--strict-topology=true"
            - "--extra-create-metadata=true"
            - "--provisioner=org.democratic-csi.iscsi-generic"
          volumeMounts:
            - name: socket-dir
              mountPath: /csi-data
      volumes:
        - name: socket-dir
          hostPath:
            # Connects directly to the active socket managed by your node daemon pods
            path: /var/lib/kubelet/plugins/org.democratic-csi.iscsi-generic
            type: DirectoryOrCreate
OF

kubectl apply -f standalone-csi-provisioner.yaml

In [ ]:
kubectl delete statefulset clickhouse-xfs --ignore-not-found=true
kubectl delete pvc xdr-data-volume-clickhouse-xfs-0 --ignore-not-found=true

# Re-apply the ClickHouse test file
kubectl apply -f clickhouse-xfs-test.yaml